# STEP 3 — Klasifikasi stres termal UTCI maksimum Jawa (1996–2025)
Replikasi metode Step 3 Sulawesi, sekarang memakai file daily max Jawa dari Step 2.

**Kategori standar:** 1: extreme cold (`<−40`), 2: very strong cold (`−40…<−27`), 3: strong cold (`−27…<−13`), 4: moderate cold (`−13…<0`), 5: slight cold (`0…<9`), 6: no stress (`9…<26`), 7: moderate heat (`26…<32`), 8: strong heat (`32…<38`), 9: very strong heat (`38…≤46`), 10: extreme heat (`>46` °C).

Statistik tahunan termasuk kategori eksklusif, hitungan kumulatif ≥26/32/38 °C, UTCI mean dan UTCI max. **QC memakai seluruh 10 kelas**, bukan asumsi kelas panas saja.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

OUT_DIR=Path(r"D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025")
DAILY=OUT_DIR/'03_utci_dailymax_1996_2025_C_Jawa.nc'
CLASS_FILE=OUT_DIR/'05_utci_thermal_stress_class_daily_1996_2025.nc'
ANNUAL_FILE=OUT_DIR/'06_utci_thermal_stress_annual_1996_2025.nc'
CSV_FILE=OUT_DIR/'07_utci_thermal_stress_annual_spatial_mean.csv'
START_YEAR,END_YEAR=1996,2025
FILL_CLASS=np.int8(-127)
if not DAILY.exists():raise FileNotFoundError(f'Hasil STEP 2 belum ada: {DAILY}')

## 1. Baca dan QC data UTCI harian Jawa

In [2]:
with xr.open_dataset(DAILY,decode_timedelta=False) as ds:
    if 'utci_max' not in ds:raise KeyError('Variabel utci_max tidak ditemukan')
    da=ds.utci_max.transpose('time','latitude','longitude')
    dates=pd.DatetimeIndex(da.time.values)
    lat=da.latitude.values.astype(float);lon=da.longitude.values.astype(float)
    units=str(da.attrs.get('units','')).lower()
    values=np.asarray(da.values,dtype='float32').copy()
expected=pd.date_range(f'{START_YEAR}-01-01',f'{END_YEAR}-12-31',freq='D')
if not dates.equals(expected):raise ValueError('Tanggal dataset daily UTCI tidak lengkap/urut.')
if units not in ('degc','c','celsius','degree_celsius','degrees_celsius'):
    raise ValueError(f'Unit daily UTCI tidak diketahui: {units}; seharusnya degC.')
valid=np.isfinite(values); land=valid.any(axis=0)
if not land.any():raise ValueError('Tidak ada grid Jawa yang valid.')
print('Shape:', values.shape,'| valid grid:',int(land.sum()),'| min/max:',float(np.nanmin(values)),float(np.nanmax(values)))

Shape: (10958, 12, 36) | valid grid: 176 | min/max: 19.581024169921875 45.90130615234375


## 2. Klasifikasi 10 kategori dan simpan file harian

In [3]:
# np.digitize(..., right=False): batas kiri masuk kategori berikutnya.
# UTCI = 46 °C tetap kategori 9, sehingga titik 46 diberi penanganan khusus.
edges=np.array([-40,-27,-13,0,9,26,32,38,46],dtype=float)
cls=np.full(values.shape,FILL_CLASS,dtype='int8')
cls[valid]=np.digitize(values[valid],edges,right=False).astype('int8')+1
cls[valid & (values==46.0)]=9
if np.any(valid & ((cls<1)|(cls>10))):raise RuntimeError('Ada data valid di luar 10 kategori.')

class_da=xr.DataArray(cls,coords={'time':dates,'latitude':lat,'longitude':lon},
                      dims=('time','latitude','longitude'),name='utci_stress_class',
                      attrs={'long_name':'UTCI thermal stress class from daily maximum','flag_values':np.arange(1,11,dtype='int8'),
                             'flag_meanings':'extreme_cold very_strong_cold strong_cold moderate_cold slight_cold no_stress moderate_heat strong_heat very_strong_heat extreme_heat'})
ctmp=CLASS_FILE.with_name(CLASS_FILE.stem+'.part.nc')
class_da.to_dataset().to_netcdf(ctmp,engine='netcdf4',encoding={'utci_stress_class':{'_FillValue':int(FILL_CLASS),'zlib':True,'complevel':3}})
os.replace(ctmp,CLASS_FILE)
print('Klasifikasi tersimpan:',CLASS_FILE)

Klasifikasi tersimpan: D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025\05_utci_thermal_stress_class_daily_1996_2025.nc


## 3. Statistik tahunan dan QC kategori lengkap

In [4]:
CATEGORY_NAMES={1:'Extreme cold',2:'Very strong cold',3:'Strong cold',4:'Moderate cold',5:'Slight cold',
                6:'No thermal stress',7:'Moderate heat',8:'Strong heat',9:'Very strong heat',10:'Extreme heat'}
varnames={1:'ECS_days',2:'VSCS_days',3:'SCS_days',4:'MCS_days',5:'SLCS_days',
          6:'NTS_days',7:'MHS_days',8:'SHS_days',9:'VSHS_days',10:'EHS_days'}
years=np.arange(START_YEAR,END_YEAR+1)
ny,nlat,nlon=len(years),len(lat),len(lon)
metrics=list(varnames.values())+['HeatStress_days','StrongPlus_days','VeryStrongPlus_days','UTCI_mean','UTCI_max']
annual={k:np.full((ny,nlat,nlon),np.nan,dtype='float32') for k in metrics}
for yi,year in enumerate(years):
    selection=(dates.year==year)
    vals=values[selection]; categories=cls[selection]; v=np.isfinite(vals)
    count=v.sum(axis=0); grid=count>0
    for code,name in varnames.items():
        z=(categories==code).sum(axis=0).astype('float32');z[~grid]=np.nan
        annual[name][yi]=z
    for name,flag in [('HeatStress_days',26),('StrongPlus_days',32),('VeryStrongPlus_days',38)]:
        z=(v & (vals>=flag)).sum(axis=0).astype('float32');z[~grid]=np.nan
        annual[name][yi]=z
    # Where grid has data: divide sum by count; avoids all-NaN warnings.
    sums=np.nansum(vals,axis=0)
    mean=np.divide(sums,count,out=np.full_like(sums,np.nan,dtype='float32'),where=count>0)
    maximum=np.max(np.where(v,vals,-np.inf),axis=0);maximum[~grid]=np.nan
    annual['UTCI_mean'][yi]=mean;annual['UTCI_max'][yi]=maximum
    total_categories=sum((categories==i).sum(axis=0) for i in range(1,11))
    if not np.array_equal(total_categories,count):
        raise RuntimeError(f'{year}: total seluruh 10 kelas != jumlah hari valid')
    if yi%5==0 or yi==len(years)-1:print(f'QC & statistik {year}: OK ({int(grid.sum())} grid)')
print('✅ QC klasifikasi: seluruh tahun LULUS')

QC & statistik 1996: OK (176 grid)
QC & statistik 2001: OK (176 grid)
QC & statistik 2006: OK (176 grid)
QC & statistik 2011: OK (176 grid)
QC & statistik 2016: OK (176 grid)
QC & statistik 2021: OK (176 grid)
QC & statistik 2025: OK (176 grid)
✅ QC klasifikasi: seluruh tahun LULUS


## 4. Simpan annual NetCDF dan area-weighted CSV

In [5]:
annual_ds=xr.Dataset({k:(('year','latitude','longitude'),v) for k,v in annual.items()},
                     coords={'year':years,'latitude':lat,'longitude':lon},
                     attrs={'title':'UTCI annual thermal stress over Jawa','period':f'{START_YEAR}–{END_YEAR}',
                            'method':'all 10 UTCI stress categories, from daily maximum'})
for key in metrics:annual_ds[key].attrs['units']='degC' if key.startswith('UTCI_') else 'days'
annual_tmp=ANNUAL_FILE.with_name(ANNUAL_FILE.stem+'.part.nc')
annual_ds.to_netcdf(annual_tmp,engine='netcdf4',encoding={k:{'zlib':True,'complevel':3,'_FillValue':np.float32(-9999)} for k in metrics})
os.replace(annual_tmp,ANNUAL_FILE)
weights=np.cos(np.deg2rad(lat))[:,None]*np.ones((1,len(lon)))
weights=np.where(land,weights,0)
def regional_mean(a):
    a=np.asarray(a,float)
    mask=np.isfinite(a) & (weights>0)
    return float(np.sum(a[mask]*weights[mask])/weights[mask].sum()) if np.any(mask) else np.nan
rows=[{'year':int(year),**{k:regional_mean(annual[k][yi]) for k in metrics}} for yi,year in enumerate(years)]
summary=pd.DataFrame(rows);summary.to_csv(CSV_FILE,index=False)
display(summary.head())
print('✅ STEP 3 LULUS:',ANNUAL_FILE,'\n',CSV_FILE)

,year,ECS_days,VSCS_days,SCS_days,MCS_days,SLCS_days,NTS_days,MHS_days,SHS_days,VSHS_days,EHS_days,HeatStress_days,StrongPlus_days,VeryStrongPlus_days,UTCI_mean,UTCI_max
0,1996,0.0,0.0,0.0,0.0,0.0,1.226530,75.864141,263.810076,25.099254,0.0,364.773470,288.909329,25.099254,34.133395,38.072275
1,1997,0.0,0.0,0.0,0.0,0.0,3.855685,82.122907,250.573578,28.447830,0.0,361.144315,279.021408,28.447830,34.027668,38.451695
2,1998,0.0,0.0,0.0,0.0,0.0,0.346698,42.638370,284.937246,37.077686,0.0,364.653302,322.014932,37.077686,34.999313,38.453789
3,1999,0.0,0.0,0.0,0.0,0.0,2.321316,88.800996,254.069901,19.807787,0.0,362.678684,273.877688,19.807787,33.806603,37.709720
4,2000,0.0,0.0,0.0,0.0,0.0,2.298309,84.081704,259.959443,19.660544,0.0,363.701691,279.619987,19.660544,33.879926,37.783956


✅ STEP 3 LULUS: D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025\06_utci_thermal_stress_annual_1996_2025.nc 
 D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025\07_utci_thermal_stress_annual_spatial_mean.csv
